# backward-on-scalar-loss — worked example 1: Reduce per-sample loss with .sum() before backward()

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `backward-on-scalar-loss`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`.backward()` only works on a **scalar** tensor. Given a per-sample loss vector you must reduce it first. `.sum()` is a valid reduction: the gradient of `sum(L_i)` w.r.t. a leaf is just the sum of each `dL_i/d(leaf)`, so each sample contributes its raw gradient (no `1/N` factor, unlike `.mean()`).

## Worked solution

1. **Build the per-sample loss.** `per_sample_loss = (w * x - y) ** 2` is element-wise, so it has the same shape as `x` (here shape `(4,)`). This is a vector, not a scalar.
2. **Why we can't call `.backward()` yet.** Calling `.backward()` on a non-scalar raises `RuntimeError: grad can be implicitly created only for scalar outputs`, because autograd needs to know the upstream gradient for each output element. A scalar implicitly seeds that with `1.0`.
3. **Reduce with `.sum()`.** `scalar_loss = per_sample_loss.sum()` collapses the vector to a 0-dim scalar. Mathematically `L = \sum_i (w x_i - y_i)^2`.
4. **Call `scalar_loss.backward()`.** Autograd seeds the scalar output with `1.0` and walks the graph back to the leaf `w`. The analytic gradient is `dL/dw = \sum_i 2 (w x_i - y_i) x_i`. Note there is **no division by N** because we summed.
5. **Read `w.grad`.** After backward, the leaf tensor `w` has its `.grad` field populated. We return the scalar and the grad so the caller can verify both.

In [ ]:
def backward_sum(w, x, y):
    per_sample_loss = (w * x - y) ** 2
    scalar_loss = per_sample_loss.sum()
    scalar_loss.backward()
    return scalar_loss, w.grad

t.manual_seed(0)
w = t.tensor([2.0], requires_grad=True)
x = t.tensor([1.0, 2.0, 3.0, 4.0])
y = t.tensor([0.5, 1.0, 2.0, 3.0])
loss, grad = backward_sum(w, x, y)
print('scalar loss:', loss.item())
print('w.grad:', grad)
# analytic check: sum(2*(w*x - y)*x)
expected = (2 * (2.0 * x - y) * x).sum()
print('expected grad:', expected.item())